Data Analyst: Andrei Berner Arroyo Tanta

**Project with Numpy and Pandas Final: Monte Cralo with GBM simulation

**Executive Summary:

-Key Findings: The system successfully processed 1,781 financial records, optimizing memory usage through float32 transformation.

-Relevant Metrics: Identified top performers like SWKS (Solvency Ratio: 5.56) and flagged high-risk entities like KMX and HCA with critical liquidity deficits.

-Risks Detected: Automated categorization of "At Risk (Underfunded)" companies based on a threshold of 1.0, enabling proactive intervention.

In [15]:
#Setting Enviroment
import numpy as np
import pandas as pd

In [16]:
# Configuration of parameters
np.random.seed(42) # Reproducibility
scenarios = 10000
months = 12
initial_flow = 100000 # Example: Starting with $100,000 USD
mu = 0.02 # Expected growth (2%)
sigma = 0.07 # Volatility (7%)
dt = 1 # Time step (monthly)

In [17]:
#Geometric Brownian Motion (GBM) Simulation (Numpy Layer)

#Generate random shocks based on a standard normal distribution Z - N(0,1)
z = np.random.normal(0, 1, (scenarios, months))

#Calculate log returns: (mu - 0.5 * sigma^2) * dt + sigma * sqrt(dt) * Z
#Using formula for GBM log-normal growth
log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
cumulative_returns = np.exp(np.cumsum(log_returns, axis=1))

#Multiply by initial flow to get cash flow trajectories
trajectories = initial_flow * cumulative_returns

#Insert month 0 (initial value) for all scenarios
start = np.full((scenarios, 1), initial_flow)
full_data = np.concatenate((start, trajectories), axis=1)

In [18]:
#Create large DataFrame
columns = [f"Month {i}" for i in range(months + 1)]
df = pd.DataFrame(full_data, columns=columns)

#Identify "Bankruptcy" Scenarios (Flow < 0)
#Create a boolean mask to detect if flow drops below zero in any month
bankruptcy_mask = (df < 0).any(axis=1)
total_bankruptcies = bankruptcy_mask.sum()
prob_bankruptcy = (total_bankruptcies / scenarios) * 100

#Calculate Value at Risk (VaR) at 5%
#VaR represents the 5th percentile of worst outcomes in the final month
var_5 = df["Month 12"].quantile(0.05)

#Identify worst case scenario (Black Swan)
worst_scenario_value = df["Month 12"].min()
#Identify best case scenario
best_scenario_value = df["Month 12"].max()

In [24]:
#Report
print(f"--- Inventory Report ---")
print(f"Probability of Bankruptcy: {np.round(prob_bankruptcy,2)}%")
print(f"Value at Risk (VaR 5%): ${np.round(var_5,2)} USD")
print(f"Worst case scenario result: ${np.round(worst_scenario_value,2)} USD")
print(f"Percentage of worst scenario: {((worst_scenario_value - initial_flow) / initial_flow):.2%}")
print(f"Worst case scenario result: ${np.round(best_scenario_value,2)} USD")
print(f"Percentage of best scenario: {((best_scenario_value - initial_flow) / initial_flow):.2%}")

--- Inventory Report ---
Probability of Bankruptcy: 0.0%
Value at Risk (VaR 5%): $82720.04 USD
Worst case scenario result: $54609.71 USD
Percentage of worst scenario: -45.39%
Worst case scenario result: $306845.49 USD
Percentage of best scenario: 206.85%
